In [1]:
import zipfile

In [4]:
output_dir = 'peft_output'
with zipfile.ZipFile('peft_output.zip','r') as f:
    f.extractall(output_dir)

In [5]:
%pip install --upgrade pip
#%pip uninstall transformers -y
#%pip uninstall torch torchvision torchaudio -y
#%pip cache purge
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install transformers==4.52.4
%pip install accelerate peft bitsandbytes datasets sentencepiece tokenizers safetensors trl
from transformers import AutoModelForCausalLM, AutoTokenizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 14.7 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 29.3 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 45.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 143.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.8/799.8 kB 57.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [transformers] [transformers]ub]
Note: you may need to restart the kernel to use updated packages.
INFO: pip is looking at multiple versions of tr

In [8]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
output_dir = 'peft_output/peft_output'
use_4bit = False
model_name = 'Qwen/Qwen2.5-0.5B-Instruct'

from peft import PeftModel
reload_kwargs = {'trust_remote_code':True}
if use_4bit:
    reload_kwargs.update({'load_in_4bit' : True, 'device_map':'auto'})
else:
    reload_kwargs.update({
        'dtype':torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    })
base_model_for_eval = AutoModelForCausalLM.from_pretrained(model_name, **reload_kwargs)    
model = PeftModel.from_pretrained(base_model_for_eval,output_dir)
if torch.cuda.is_available() and not use_4bit:
    model.to('cuda')
model.eval()
print('reload PEFT adapter from:',output_dir)
print('Eval model dtype:',next(model.parameters()).dtype)


reload PEFT adapter from: peft_output/peft_output
Eval model dtype: torch.bfloat16


In [10]:
import random
LABEL_DESC = {
    'DEF': '정의/목적/적용범위 조항',
    'RIGHT': '권리/의무/금지/책임 조항',
    'PROC': '신청/심사/조사/불복/처벌 절차 조항',
    'ORG': '기관/위원회/법원 등 조직의 설치/구성/권한 조항',
    'CRIT': '자격/요건/기준/기간/수치 조건 조항',
    'ETC': '시행일/경과조치/위임 등 기타 조항',
}

label_list = list(LABEL_DESC.keys())

valid_file = r'./validation.jsonl'

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

def predict_label_from_prompt(prompt_text, max_new_tokens=6):
    model.eval()
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    generated = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

    upper = generated.upper()
    for lb in label_list:
        if lb in upper:
            return lb, generated

    first = generated.split()[0].strip(".,:;()[]{}\"'").upper() if generated else ''
    if first in label_list:
        return first, generated
    return 'ETC', generated


val_ds = load_dataset('json', data_files={'validation': valid_file})['validation']
num_eval = min(100, len(val_ds))
indices = random.sample(range(len(val_ds)), k=num_eval)

correct = 0
samples = []
for idx in indices:
    item = val_ds[idx]
    pred, raw = predict_label_from_prompt(item['prompt'])
    gt = item['response']
    correct += int(pred == gt)
    if len(samples) < 5:
        samples.append((gt, pred, raw))

acc = correct / num_eval if num_eval else 0.0
print(f'Validation sample accuracy ({num_eval}개): {acc:.4f}')
print('\n예시 5개 (정답, 예측, 생성원문):')
for row in samples:
    print(row)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Validation sample accuracy (100개): 1.0000

예시 5개 (정답, 예측, 생성원문):
('DEF', 'DEF', 'DEF(정의/목')
('RIGHT', 'RIGHT', 'RIGHT(의무)')
('DEF', 'DEF', 'DEF(정의/목')
('DEF', 'DEF', 'DEF(정의/목')
('RIGHT', 'RIGHT', 'RIGHT(권리)')
